### Decoding Notes

Decode 1000 cameras × 10 FPS = **10,000 decoded frames per second**, each frame is 1920×1080 (see the example from VIRAT)

H200 have 7 dedicated NVDEC (Hardware-accelerated decoding) components

there are no official numbers published for H200 on decoding but [NVIDIA decoder SDK](https://docs.nvidia.com/video-technologies/video-codec-sdk/12.2/pdf/NVDEC_Application_Note.pdf) document gives per-NVDEC HEVC (H.265) throughput measured at 1920×1080/YUV420 for various other GPU families (can consider RTX8000)

Turing (RTX8000) — HEVC ≈ 1247 fps per NVDEC

7 × 1247 = 8,729 fps = 
GPU = 10,000 / 8,729 = 1.14 (RTX8000)

Probably 0.5 H200?

### Object Detection Notes

* NVIDIA H200, decoding 1000 camera streams (10k FPS)
* YOLOE-L @ 640×640: **86.9 GFLOPs/frame** (i.e. 86.9 × 10⁹ ops)
* Total throughput: **10,000 frames/sec**
* H200 FP16 peak: **≈ 1,979 TFLOPs = 1,979 × 10¹² ops/sec**

### compute required

$$
C_{\text{total}} = 86.9 \times 10^{9} ; \text{ops/frame} \times 10{,}000 ; \text{frames/s}
= 8.69 \times 10^{14} ; \text{ops/s}
$$

TFLOPs:

$$
C_{\text{total}} = \frac{8.69 \times 10^{14}}{10^{12}} = 869 ;\text{TFLOPs}
$$

### H200 GPUs (theoretical, FP16)

$$
N_{\text{H200}} = \frac{869}{1,979} \approx 0.44
$$

https://docs.ultralytics.com/models/yoloe/#launching-training-from-scratch

In [ ]:
ops_per_frame = 86.9e9 # from yoloe page on T4 hardware
frames_per_second = 10000 # 1K CAMERAs
tera_conversion_factor = 1e12
h200_theoretical_flops = 1979 # from NVIDIA spec sheet

c_total_ops_per_second = ops_per_frame * frames_per_second
print(f"C_total in ops/s: {c_total_ops_per_second}")
c_total_tflops = c_total_ops_per_second / tera_conversion_factor
print(f"C_total in TFLOPs: {c_total_tflops}")
n_h200 = c_total_tflops / h200_theoretical_flops
print(f"Number of H200 GPUs needed (N_H200): {n_h200}")



C_total in ops/s: 869000000000000.0
C_total in TFLOPs: 869.0
Number of H200 GPUs needed (N_H200): 0.43911066195048004


In [ ]:
import math

def compute_gpu_requirements(
    gflops_fp32_per_patch,
    patches_per_frame,
    fps,
    h200_fp32_tflops=989
):
    per_frame_tflops = (gflops_fp32_per_patch * patches_per_frame) / 1000
    total_tflops = per_frame_tflops * fps
    h200_required = total_tflops / h200_fp32_tflops
    return per_frame_tflops, total_tflops, h200_required


gflops_fp32 = 173.8         
patches = 8                 
fps = 10_000                
h200_fp32 = 989            

per_frame_tflops, total_tflops, h200_required = compute_gpu_requirements(
    gflops_fp32,
    patches,
    fps,
    h200_fp32
)

print("\n===== SAHI GPU with 8x PATCHES =====")
print(f"Per 1920×1080 frame compute : {per_frame_tflops:.2f} TFLOPs")
print(f"Total compute for {fps:,} FPS : {total_tflops:.2f} TFLOPs")
print(f"H200 : {math.ceil(h200_required)} (raw={h200_required:.2f})")
print("========================================\n")



===== SAHI GPU with 8x PATCHES =====
Per 1920×1080 frame compute : 1.39 TFLOPs
Total compute for 10,000 FPS : 13904.00 TFLOPs
H200 : 15 (raw=14.06)



Yolo11 default is FP32

For FP32:

$$
C_{\text{FP32}} = 2 \times 86.9 = 173.8\ \text{GFLOPs/frame}
$$


$$
C_{\text{total}} = 173.8\ \text{GFLOPs} \times 10{,}000
$$

$$
C_{\text{total}} = 1{,}738{,}000\ \text{GFLOPs/s} = 1{,}738\ \text{TFLOPs}
$$

### h200

$$
N = \frac{1{,}738}{989} \approx 1.76 \text{H200}
$$

### SAHI (Slicing Aided Hyper Inference)

SAHI overhead with default FP32 precision. The frame size is 1920 × 1080, and SAHI creates 8 patches with below configuration 
- slice_height=640
- slice_width=640 
- overlap_height_ratio=0.2 
- overlap_width_ratio=0.2

**Compute per original frame with SAHI**

$$
C_{\text{SAHI per frame}} = 8 \times 173.8\ \text{GFLOPs}
= 1,390.4\ \text{GFLOPs}
= 1.39\ \text{TFLOPs per 1920×1080 frame}
$$

* Baseline (no SAHI, just resize to 640×640): 173.8 GFLOPs/frame
* With SAHI: 1390.4 GFLOPs per frame

$$
\text{Overhead factor} = \frac{1,390.4}{173.8} = 8\times
$$

**With SAHI enabled (8 patches per frame, FP32):**

$$
1.39 \times 10{,}000 = 13{,}900\ \text{TFLOPs}
$$

* H200 FP32 = 989 TFLOPs

$$
\frac{13{,}900}{989} \approx 14.07 \text{ H200}
$$

### Ground Truth from NVIDIA test

8 × H100 running VILA 34B (TRT-LLM INT4 AWQ) 
- 220 parallel live streams total (≈27 streams / GPU) at 94% avg GPU util
- 10-s chunks, max 100 output tokens, avg latency 9.6 s. (your data — used directly).

H200 specs (reference): FP8/INT8 and FP16 TFLOPS comparable to H100; H200 has 141 GB HBM3e and 4.8 TB/s memory bandwidth.